# Astra — word-level base + BIG DailyDialog chat fine-tune (Kaggle GPU)

This is the v2 Kaggle notebook. It fixes the rambling chat output by training on a **big multi-turn corpus** instead of the 5 MB hand-authored one.

Stages (about 25–35 min total):
1. **DailyDialog** (~13k multi-turn dialogues) → reframed as `You:` / `Astra:` turns, merged with Astra's identity pairs, n=13 leak gate
2. **word tokenizer** rebuilt over prose + the new big chat corpus (16,384 words, far fewer `<unk>`)
3. **word-prose** 5400 steps (English base)
4. **word-chat** 3800 steps warm-started from prose final

**Before running:** the corpus builder `datasets/chat/make_corpus_dailydialog.py` must exist in the cloned repo. This notebook clones from GitHub, so push it first (see the check in the next code cell).

Runtime setup: **Settings → Accelerator: GPU (T4)**, Internet: ON (required for git clone and the DailyDialog download).

**Persistence:** commit the version; everything staged in `/kaggle/working/output` is saved to the version's **Output** tab.

In [ ]:
import torch
print('torch', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0), torch.cuda.get_device_capability(0))

**Preflight** — this notebook needs a GPU **and** Internet. If this cell stops, flip both Kaggle settings and re-run:

- **Settings → Accelerator → GPU (T4)** (otherwise `torch ...+cpu`, `cuda available: False`)
- **Settings → Internet → On** (otherwise `Could not resolve host: github.com` and the clone + DailyDialog download both fail)

In [ ]:
import socket, sys

fails = []

if not torch.cuda.is_available():
    fails.append('No GPU: Settings → Accelerator → GPU (T4). You have torch ' + torch.__version__)

try:
    socket.gethostbyname('github.com')
except OSError:
    fails.append('No Internet: Settings → Internet → On. (Needed for the git clone AND the DailyDialog download.)')

if fails:
    print('=' * 60)
    for f in fails:
        print('  ✗', f)
    print('=' * 60)
    raise SystemExit(1)

print('preflight OK: gpu + internet')

In [ ]:
import os, subprocess

REPO = '/kaggle/working/astra'
if os.path.isdir(REPO):
    subprocess.run(f'rm -rf {REPO}', shell=True, check=True)
subprocess.run(f'git clone --depth 1 https://github.com/anoneurx/astra.git {REPO}', shell=True, check=True)
assert os.path.isdir(REPO), 'clone failed - check Internet is ON and retry'
os.chdir(REPO)
print('cwd:', os.getcwd())

BUILDER = 'datasets/chat/make_corpus_dailydialog.py'
if not os.path.exists(BUILDER):
    print('=' * 60)
    print('MISSING:', BUILDER)
    print('Commit and push that file to GitHub, then re-run this cell.')
    print('=' * 60)
    raise SystemExit(1)
print('corpus builder present')

---
**Stage 1 — download DailyDialog** (HF, ~13k dialogues). Writes `datasets/chat/dailydialog_cleaned.jsonl`, one `{"dialog": [...]}` per line.

In [ ]:
!pip install -q datasets

import json, os
from datasets import load_dataset

OUT = 'datasets/chat/dailydialog_cleaned.jsonl'
os.makedirs('datasets/chat', exist_ok=True)

ds = load_dataset('daily_dialog')
split = 'train' if 'train' in ds else list(ds.keys())[0]
rows = ds[split]
print('split:', split, 'dialogues:', len(rows))

with open(OUT, 'w', encoding='utf-8') as f:
    for r in rows:
        d = r.get('dialog') or []
        if len(d) < 2:
            continue
        f.write(json.dumps({'dialog': list(d)}, ensure_ascii=False) + '\n')

print('wrote', OUT, os.path.getsize(OUT), 'bytes')

**Stage 2 — build the BIG chat corpus** (reframes turns as `You:` / `Astra:`, merges identity pairs, runs the n=13 contamination gate, writes `datasets/chat/train.txt` + `val.txt`).

In [ ]:
!python datasets/chat/make_corpus_dailydialog.py --src datasets/chat/dailydialog_cleaned.jsonl 2>&1 | tail -20

!wc -c datasets/chat/train.txt datasets/chat/val.txt

Expect `train.txt` to be **much larger** than the old 5,152,375 bytes. The leak gate should print `leak_free`.

---
**Stage 3 — rebuild the word tokenizer** over prose + the new big chat corpus. This is what pushes everyday conversational words into the 16,384-word vocab and cuts `<unk>` at inference.

In [ ]:
import os, json

if os.path.exists('tokenizer/artifacts/prose_chat_word.json'):
    os.remove('tokenizer/artifacts/prose_chat_word.json')
    print('removed old tokenizer artifact (forced rebuild over the new corpus)')

!python tools/build_word_tokenizer.py 2>&1 | tail -20

a = json.load(open('tokenizer/artifacts/prose_chat_word.json'))
print('tokenizer vocab:', a['vocab_size'])

**Smoke test** — 10 steps of word-prose to verify the pipeline on GPU before the real run.

In [ ]:
!mkdir -p /kaggle/working/runs

!python training/gpu_train.py --config configs/astra5m_word_prose.json \
  --steps 10 --out /kaggle/working/runs/word_prose_smoke \
  --cache-dir /kaggle/working/cache 2>&1 | tail -8

Sanity: last line should show `[torch] device=cuda`, a `[step ...]` line, and `checkpoint -> ...final.npz`. Then the JSONL token cache is built, so the real runs below are fast.

---
**Stage 4/5 — word-prose base (5400 steps)**

Look for `[step 5400]` and `checkpoint -> /kaggle/working/runs/word_prose/final.npz`.

In [ ]:
!python training/gpu_train.py --config configs/astra5m_word_prose.json \
  --steps 5400 --out /kaggle/working/runs/word_prose \
  --cache-dir /kaggle/working/cache 2>&1 | tail -20
print('=== prose final ===')
!ls -la /kaggle/working/runs/word_prose/final.npz

**Stage 5/5 — word-chat fine-tune on the BIG DailyDialog corpus (3800 steps)**, warm-started from the prose final.

Look for `[step 3800]`, the chat val perplexity, and `checkpoint -> /kaggle/working/runs/word_chat/resumed/final.npz`.

In [ ]:
!python training/gpu_train.py --config configs/astra5m_word_chat.json \
  --steps 3800 --out /kaggle/working/runs/word_chat \
  --resume /kaggle/working/runs/word_prose/final.npz --reset-step \
  --cache-dir /kaggle/working/cache 2>&1 | tail -25
print('=== chat final ===')
!ls -la /kaggle/working/runs/word_chat/resumed/final.npz

**Sanity-check the generation** before downloading — sample a few chat turns with the freshly fine-tuned model.

In [ ]:
import subprocess, os

CHAT = '/kaggle/working/runs/word_chat/resumed/final.npz'
print('chat final exists:', os.path.exists(CHAT))

prompts = ['what is your name', 'where are you from', 'how are you', 'tell me a joke']
stdin = '\n'.join(prompts + ['quit']) + '\n'

r = subprocess.run(
    ['python', 'inference/generate.py', '--chat',
     '--checkpoint', CHAT, '--config', 'configs/astra5m_word_chat.json',
     '--max-new', '40', '--temperature', '0.6', '--top-k', '8', '--rep-penalty', '1.15'],
    input=stdin, capture_output=True, text=True,
)
print(r.stdout[-4000:])
if r.returncode != 0:
    print('STDERR:', r.stderr[-2000:])

**Stage outputs** — everything under `/kaggle/working/output` is saved to the version's **Output** tab (persistent, downloadable from the Kaggle UI).

In [ ]:
import os, shutil, json, hashlib

out = '/kaggle/working/output'
os.makedirs(out, exist_ok=True)

pairs = [
    ('/kaggle/working/runs/word_prose/final.npz', 'astra_word_prose_final.npz'),
    ('/kaggle/working/runs/word_chat/resumed/final.npz', 'astra_word_chat_final.npz'),
]
manifest = {}
for src, dst in pairs:
    if os.path.exists(src):
        shutil.copy(src, os.path.join(out, dst))
        b = open(src, 'rb').read()
        manifest[dst] = {'bytes': len(b), 'md5': hashlib.md5(b).hexdigest()}
        print(f'[ok] {dst} ({len(b):,} bytes) md5={manifest[dst]["md5"][:12]}')
    else:
        print(f'MISSING: {src}')

if os.path.exists(TOK):
    shutil.copy(TOK, os.path.join(out, 'prose_chat_word.json'))
    print('[ok] prose_chat_word.json')

with open(os.path.join(out, 'manifest.json'), 'w') as f:
    json.dump(manifest, f, indent=2)
print('\nDownload: this notebook -> Save version -> Output tab.')
print('Local install target: checkpoints/astra5m_word_chat/resumed/final.npz')

**Optional: push to a Kaggle Dataset via API** (most durable). Requires kernel secrets `KAGGLE_USERNAME` and `KAGGLE_KEY` (Add → Secrets). Creates `astra-word-checkpoints` on first run, versions it afterwards.

Then locally: `kaggle datasets download -d <username>/astra-word-checkpoints`.

In [ ]:
import os
try:
    from kaggle_secrets import UserSecretsClient
    s = UserSecretsClient()
    os.environ['KAGGLE_USERNAME'] = s.get_secret('KAGGLE_USERNAME')
    os.environ['KAGGLE_KEY'] = s.get_secret('KAGGLE_KEY')
except Exception as e:
    print('no Kaggle API secrets configured - skip')
else:
    import json
    out = '/kaggle/working/output'
    meta = {
        'id': f"{os.environ['KAGGLE_USERNAME'].lower()}/astra-word-checkpoints",
        'title': 'Astra word checkpoints',
        'licenses': [{'name': 'MIT'}],
    }
    with open(os.path.join(out, 'dataset-metadata.json'), 'w') as f:
        json.dump(meta, f)
    r = os.system(f'kaggle datasets create -p {out}')
    if r != 0:
        os.system(f'kaggle datasets version -p {out} -m "checkpoints"')